# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as object attributes (not like a dictionary!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}\nLicense: {dataset.metadata.license}")
print(f"Citation: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list all record sets, their fields, and columns, referencing all entities by their `@id`.

In [ ]:
def print_record_sets_info(ds):
    print("Record sets available:\n")
    record_sets = list(ds.record_sets)
    for rs in record_sets:
        rs_id = rs['@id']
        name = rs.get('name', 'N/A')
        print(f"- Record Set: {name} (ID: {rs_id})")

        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  Fields:")
            for f in fields:
                f_id = f.get('@id') if isinstance(f, dict) else f
                label = f.get('name', 'N/A') if isinstance(f, dict) else 'N/A'
                print(f"    - {f_id}  (name: {label})")

        if 'column' in rs:
            cols = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
            print("  Columns:")
            for c in cols:
                c_id = c.get('@id') if isinstance(c, dict) else c
                label = c.get('name', 'N/A') if isinstance(c, dict) else 'N/A'
                print(f"    - {c_id}  (name: {label})")
        print()

# Show record sets, their fields and columns
print_record_sets_info(dataset)

## 3. Data Extraction
Let's load the data from each record set into a DataFrame for analysis.

We'll use the record set and field `@id`s listed in the overview above.


In [ ]:
# Identify all record sets
record_sets = [rs['@id'] for rs in dataset.record_sets]
# Use a dictionary to keep DataFrames for all record sets
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nLoaded record set {record_set_id}: {len(records)} records, columns: {list(dataframes[record_set_id].columns)}")
    else:
        print(f"\nRecord set {record_set_id} has no records (empty or not a data table).")

# For demonstration, select the first record set that contains records
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"\nWe'll use record set '{main_record_set_id}' for further analysis.")
    print('First few rows:')
    display(dataframes[main_record_set_id].head())
else:
    print('No record set with data found! Exploration cannot continue.')

## 4. Exploratory Data Analysis (EDA)
Let's perform some processing and transformation steps on the data, using fields referenced strictly by their `@id`s as required by the Croissant schema.


In [ ]:
import numpy as np

# You can inspect available columns (which are the field @id's, see previous cells)
df = dataframes[main_record_set_id]
print('Available columns (field @id):')
print(list(df.columns))

# For demonstration, select a numeric field by @id (customize as applicable for this dataset)
# E.g., mass spectrometry datasets may include a field such as 'cr:mol_weight' (molecular weight)
possible_numeric_fields = [c for c in df.columns if 'weight' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower()]

if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Fallback: use first numeric column detected
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.number):
            numeric_field_id = c
            break
    else:
        numeric_field_id = df.columns[0]  # just pick the first for demo

print(f"Using field '{numeric_field_id}' for numeric analysis.")

# Convert the field to numeric if not already
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Added normalized column for {numeric_field_id}.")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field (likely a categorical/label field, e.g. 'cr:sample_name' or 'cr:protein')
possible_group_fields = [c for c in df.columns if ('sample' in c.lower() or 'protein' in c.lower() or 'group' in c.lower()) and c != numeric_field_id]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    print(f"Grouped mean of filtered records by '{group_field}' (using @id):")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())
else:
    print("No suitable categorical field found to group by.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its normalized values, and show a boxplot by group if a categorical group field was detected.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)

plt.subplot(1, 2, 2)
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, color='orange')
plt.title(f"Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.tight_layout()
plt.show()

if group_field:
    plt.figure(figsize=(10,5))
    # Show top N groups if too many
    top_groups = filtered_df[group_field].value_counts().index[:10]
    sns.boxplot(data=filtered_df[filtered_df[group_field].isin(top_groups)], x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field} (top 10 groups)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the Croissant-structured dataset, explored its metadata, and examined its tabular data. Using strict referencing via the Croissant `@id` fields, we demonstrated how to filter, normalize, and visualize protein mass spectrometry data using `mlcroissant` and Python data science tools.

For deeper insights, consult the Croissant schema's field descriptions and extend the EDA to analyze additional record sets or publish downstream results.
